https://www.nature.com/articles/s41598-025-87219-w

OCT-based diagnosis of glaucoma and glaucoma stages using explainable machine learning

In [1]:
import numpy as np
from PIL import Image
import cv2
import math
import os
from tqdm import tqdm
import pandas as pd

In [2]:
df_origa = pd.read_csv(r"C:\Users\Amber\Desktop\Human-AI Colab\dataset\ORIGA\origa_info2.csv")

In [11]:
refuge_df = pd.read_csv(r"C:\Users\Amber\Desktop\Human-AI Colab\dataset\Refuge\synthetic_labels\refuge_synthetic_labels.csv")

In [3]:
df_cha = pd.read_csv(r"D:\Chaksu\chaksu.csv")

In [4]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained("pamixsun/swinv2_tiny_for_glaucoma_classification")
model = AutoModelForImageClassification.from_pretrained("pamixsun/swinv2_tiny_for_glaucoma_classification")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Swinv2ForImageClassification(
  (swinv2): Swinv2Model(
    (embeddings): Swinv2Embeddings(
      (patch_embeddings): Swinv2PatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Swinv2Encoder(
      (layers): ModuleList(
        (0): Swinv2Stage(
          (blocks): ModuleList(
            (0): Swinv2Layer(
              (attention): Swinv2Attention(
                (self): Swinv2SelfAttention(
                  (continuous_position_bias_mlp): Sequential(
                    (0): Linear(in_features=2, out_features=512, bias=True)
                    (1): ReLU(inplace=True)
                    (2): Linear(in_features=512, out_features=3, bias=False)
                  )
                  (query): Linear(in_features=96, out_features=96, bias=True)
                  (key): Linear(in_features=96, out_features=96, bias

In [6]:
def explore_swin_outputs(model, processor, path, device):
    """First, let's see what Swin V2 actually outputs."""
    model.eval()
    sample_image = Image.open(path).convert("RGB")  
    inputs = processor(images=sample_image, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
        )
    
    print("=" * 60)
    print("SWIN V2 OUTPUT STRUCTURE")
    print("=" * 60)
    
    # Logits
    print(f"\nLogits shape: {outputs.logits.shape}")
    
    # Hidden states
    print(f"\nNumber of hidden states: {len(outputs.hidden_states)}")
    for i, h in enumerate(outputs.hidden_states):
        print(f"  hidden_states[{i}]: {h.shape}")
    
    # Reshaped hidden states (Swin-specific)
    if hasattr(outputs, "reshaped_hidden_states") and outputs.reshaped_hidden_states is not None:
        print(f"\nNumber of reshaped_hidden_states: {len(outputs.reshaped_hidden_states)}")
        for i, h in enumerate(outputs.reshaped_hidden_states):
            print(f"  reshaped_hidden_states[{i}]: {h.shape}")
    
    # Attentions
    if outputs.attentions is not None:
        print(f"\nNumber of attention layers: {len(outputs.attentions)}")
        for i, a in enumerate(outputs.attentions):
            print(f"  attentions[{i}]: {a.shape}")
    
    return outputs


In [7]:
def _find_head(model):
    for attr in ("classifier", "head", "fc"):
        if hasattr(model, attr):
            return getattr(model, attr)
    raise AttributeError("Cannot find classifier head (expected .classifier / .head / .fc)")


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Swinv2ForImageClassification(
  (swinv2): Swinv2Model(
    (embeddings): Swinv2Embeddings(
      (patch_embeddings): Swinv2PatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Swinv2Encoder(
      (layers): ModuleList(
        (0): Swinv2Stage(
          (blocks): ModuleList(
            (0): Swinv2Layer(
              (attention): Swinv2Attention(
                (self): Swinv2SelfAttention(
                  (continuous_position_bias_mlp): Sequential(
                    (0): Linear(in_features=2, out_features=512, bias=True)
                    (1): ReLU(inplace=True)
                    (2): Linear(in_features=512, out_features=3, bias=False)
                  )
                  (query): Linear(in_features=96, out_features=96, bias=True)
                  (key): Linear(in_features=96, out_features=96, bias

In [15]:
import torch.nn.functional as F
@torch.inference_mode()
def extract_features(model, processor, df, batch_size=32, layers=(2, 3, 4), path_col="image_path",
                     label_col="y_true", id_col="case_id", global_id_col="global_id"):
    """Extract logits + pooled hidden states per layer. Skips unreadable images."""
    model.eval()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
 
    u = df.drop_duplicates(subset=path_col)[[id_col, path_col, label_col, global_id_col]].reset_index(drop=True)
    paths = u[path_col].tolist()

    case_ids, labels, global_ids = [], [], []
    all_logits, all_last = [], []
    all_feats = {int(l): [] for l in layers}

    for i in tqdm(range(0, len(paths), batch_size), desc="Extracting"):
        batch_idx = list(range(i, min(i + batch_size, len(paths))))
        imgs, keep = [], []
        for j, p in enumerate([paths[k] for k in batch_idx]):
            try:
                imgs.append(Image.open(p).convert("RGB"))
                keep.append(j)
            except Exception:
                pass
        if not imgs:
            continue

        outputs = model(**processor(images=imgs, return_tensors="pt").to(device),
                        output_hidden_states=True, return_dict=True)
        hs = outputs.hidden_states

        kept = [batch_idx[j] for j in keep]
        case_ids.extend(u.loc[kept, id_col].tolist())
        labels.extend(u.loc[kept, label_col].astype(int).tolist())
        global_ids.extend(u.loc[kept, global_id_col].tolist())
        all_logits.append(outputs.logits.cpu())
        all_last.append(hs[-1].mean(dim=1).cpu())
        for l in layers:
            all_feats[int(l)].append(hs[int(l)].mean(dim=1).cpu())

    if not all_logits:
        raise RuntimeError("No images were successfully processed.")

    return {
        "global_id": np.array(global_ids),
        "case_id": np.array(case_ids),
        "labels": torch.tensor(labels, dtype=torch.long),
        "logits": torch.cat(all_logits),
        "pooled_last": torch.cat(all_last),
        "hidden": {int(l): torch.cat(all_feats[int(l)]) for l in layers},
    }

In [16]:
feat_origa = extract_features(model, processor, df_origa,label_col="Label")

Extracting: 100%|██████████| 21/21 [01:33<00:00,  4.47s/it]


In [17]:
feat_cha = extract_features(model, processor, df_cha, path_col="img_path",label_col="label")

Extracting:   0%|          | 0/43 [00:00<?, ?it/s]

Extracting: 100%|██████████| 43/43 [04:25<00:00,  6.17s/it]


In [18]:
feat_refuge = extract_features(model, processor, refuge_df, label_col="y_true",)  

Extracting:   0%|          | 0/38 [00:00<?, ?it/s]

Extracting: 100%|██████████| 38/38 [02:30<00:00,  3.95s/it]


In [19]:
# Stack embeddings and global_ids
all_ids = np.concatenate([feat_refuge["global_id"], feat_cha["global_id"], feat_origa["global_id"]])
all_emb = torch.cat([feat_refuge["pooled_last"], feat_cha["pooled_last"], feat_origa["pooled_last"]], dim=0).numpy()

print(f"IDs: {len(all_ids)}, Emb: {all_emb.shape}")  # e.g. (N, 768)
assert len(all_ids) == all_emb.shape[0]
assert len(all_ids) == len(set(all_ids)), "Duplicate global_ids!"
folder_path = r"D:\Chaksu\Swinv2 emb"
# Save
np.save(f"{folder_path}/all_emb_swinv2.npy", all_emb)
np.save(f"{folder_path}/all_global_ids_swinv2.npy", all_ids)

IDs: 3195, Emb: (3195, 768)


In [6]:
def score_msp(logits):
    return (1.0 - logits.softmax(1).max(1).values).numpy()

def score_maxlogit(logits):
    return (-logits.max(1).values).numpy()

def score_entropy(logits):
    p = logits.softmax(1).clamp_min(1e-12)
    return (-(p * p.log()).sum(1)).numpy()

def score_energy(logits, T=1.0):
    return -torch.logsumexp(logits / T, dim=1).numpy()  # negate so higher = more OOD

def score_energy_react(model, feats, T=1.0, clip_c=10.0, device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    head = _find_head(model)
    with torch.inference_mode():
        logits = head(feats.to(device).clamp(max=clip_c))
    return score_energy(logits.cpu(), T)  # now inherits correct sign

In [7]:
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.covariance import LedoitWolf

class KNN_OOD:
    def __init__(self, k=10, normalize=True):
        self.k, self.normalize = k, normalize

    def _prep(self, X):
        X = np.asarray(X, dtype=np.float32)
        if self.normalize:
            X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
        return X

    def fit(self, train_feats):
        self.nn = NearestNeighbors(n_neighbors=self.k, metric="euclidean")
        self.nn.fit(self._prep(train_feats))
        return self

    def score(self, feats):
        return self.nn.kneighbors(self._prep(feats), return_distance=True)[0][:, -1]


class ViM_OOD:
    """Virtual-logit Matching: alpha * ||residual|| - logsumexp(logits)."""
    def __init__(self, pca_dim=256):
        self.pca_dim = pca_dim

    def fit(self, train_feats, train_logits):
        X = train_feats.astype(np.float32)
        self.mu = X.mean(0, keepdims=True)
        Xc = X - self.mu

        d = min(self.pca_dim, Xc.shape[0] - 1, Xc.shape[1])
        pca = PCA(n_components=d, svd_solver="full", random_state=0).fit(Xc)
        self.V = pca.components_.T  # [D, d]

        rnorm = np.linalg.norm(Xc - Xc @ self.V @ self.V.T, axis=1)
        self.alpha = float(train_logits.max(1).mean() / (rnorm.mean() + 1e-12))
        return self

    def score(self, feats, logits):
        X = feats.astype(np.float32) - self.mu
        rnorm = np.linalg.norm(X - X @ self.V @ self.V.T, axis=1)
        lse = torch.logsumexp(torch.as_tensor(logits, dtype=torch.float32), dim=1).numpy()
        return self.alpha * rnorm - lse


class Mahalanobis_OOD:
    """Multi-layer Mahalanobis with Ledoit-Wolf shrinkage."""
    def __init__(self, layers=(2, 3, 4), n_classes=2):
        self.layers = tuple(int(x) for x in layers)
        self.n_classes = n_classes
        self.means, self.icov = {}, {}

    def fit(self, feats_by_layer, y):
        y = y.astype(int)
        for l in self.layers:
            X = feats_by_layer[l].astype(np.float32)
            self.means[l] = np.stack([X[y == c].mean(0) for c in range(self.n_classes)])
            cov = LedoitWolf().fit(X).covariance_.astype(np.float32)
            cov += 1e-6 * np.eye(X.shape[1], dtype=np.float32)
            self.icov[l] = np.linalg.inv(cov)
        return self

    def score(self, feats_by_layer):
        total = None
        for l in self.layers:
            X = feats_by_layer[l].astype(np.float32)
            diff = X[:, None, :] - self.means[l][None, :, :]  # [N, C, d]
            dists = np.einsum("ncd,de,nce->nc", diff, self.icov[l], diff).min(axis=1)
            total = dists if total is None else total + dists
        return total

In [8]:

def ood_metrics(id_scores, ood_scores, tau=99.0):
    y = np.concatenate([np.zeros(len(id_scores), dtype=int),
                        np.ones(len(ood_scores), dtype=int)])
    s = np.concatenate([id_scores, ood_scores]).astype(np.float64)

    tau = np.percentile(id_scores, tau)
    accuracy_score = np.mean(id_scores <= tau)  # ID should be below threshold
    return {
        "AUROC": float(roc_auc_score(y, s)),
        "AUPR_OOD": float(average_precision_score(y, s)),
        "FPR99": float(np.mean(ood_scores <= tau)),
        "tau@99ID": float(tau),
    }

In [53]:
# ── Run OOD pipeline from pre-extracted features ─────────
LAYERS   = (2, 3, 4)
ENERGY_T = 1.0
REACT_PCTL = 90.0
KNN_K    = 10
VIM_DIM  = 256

# Split REFUGE
split = feat_refuge["split"]
idx   = {s: np.where(split == s)[0] for s in ("train", "val", "test")}

logits = {s: feat_refuge["logits"][idx[s]] for s in idx}
hlast  = {s: feat_refuge["pooled_last"][idx[s]] for s in idx}
y_tr   = feat_refuge["labels"][idx["train"]].numpy()


### OOD ###
ood_sets = {
    "origa":  feat_origa,
    "chaksu": feat_cha,        # fixed: was feat_chat
}
# Fit detectors on train
clip_c = float(np.percentile(hlast["train"].numpy().reshape(-1), REACT_PCTL))
knn    = KNN_OOD(k=KNN_K).fit(hlast["train"].numpy())
vim    = ViM_OOD(pca_dim=VIM_DIM).fit(hlast["train"].numpy(), logits["train"].numpy())

fl_tr = {l: feat_refuge["hidden"][l][idx["train"]].numpy() for l in LAYERS}
maha  = Mahalanobis_OOD(layers=LAYERS).fit(fl_tr, y_tr)

# ── Helper: extract Mahalanobis layer features ──
def _maha_feats(src, idxs=None):
    if idxs is not None:
        return {l: src["hidden"][l][idxs].numpy() for l in LAYERS}
    return {l: src["hidden"][l].numpy() for l in LAYERS}

# ── Score all splits (now including train) ──
refuge_splits = ("train", "val", "test")
scores = {}

scores["MSP"] = {
    **{("refuge", s): score_msp(logits[s]) for s in refuge_splits},
    **{(name, "all"): score_msp(feat["logits"]) for name, feat in ood_sets.items()},
}

# --- MaxLogit ---
scores["MaxLogit"] = {
    **{("refuge", s): score_maxlogit(logits[s]) for s in refuge_splits},
    **{(name, "all"): score_maxlogit(feat["logits"]) for name, feat in ood_sets.items()},
}

# --- Entropy ---
scores["Entropy"] = {
    **{("refuge", s): score_entropy(logits[s]) for s in refuge_splits},
    **{(name, "all"): score_entropy(feat["logits"]) for name, feat in ood_sets.items()},
}

# --- Energy ---
scores["Energy"] = {
    **{("refuge", s): score_energy(logits[s], ENERGY_T) for s in refuge_splits},
    **{(name, "all"): score_energy(feat["logits"], ENERGY_T) for name, feat in ood_sets.items()},
}

# --- Energy + ReAct ---
scores["Energy+ReAct"] = {
    **{("refuge", s): score_energy_react(model, hlast[s], ENERGY_T, clip_c, device) for s in refuge_splits},
    **{(name, "all"): score_energy_react(model, feat["pooled_last"], ENERGY_T, clip_c, device)
       for name, feat in ood_sets.items()},
}

# --- kNN ---
scores["kNN"] = {
    **{("refuge", s): knn.score(hlast[s].numpy()) for s in refuge_splits},
    **{(name, "all"): knn.score(feat["pooled_last"].numpy()) for name, feat in ood_sets.items()},
}

# --- ViM ---
scores["ViM"] = {
    **{("refuge", s): vim.score(hlast[s].numpy(), logits[s].numpy()) for s in refuge_splits},
    **{(name, "all"): vim.score(feat["pooled_last"].numpy(), feat["logits"].numpy())
       for name, feat in ood_sets.items()},
}

# --- Mahalanobis ---
scores["Mahalanobis"] = {
    **{("refuge", s): maha.score(_maha_feats(feat_refuge, idx[s])) for s in refuge_splits},
    **{(name, "all"): maha.score(_maha_feats(feat)) for name, feat in ood_sets.items()},
}

In [54]:


# ── Build DataFrame ──
rows = []

# REFUGE train, valid, test
for s in refuge_splits:
    ix = idx[s]
    n = len(ix)
    case_ids = feat_refuge["case_id"][ix] if "case_id" in feat_refuge else np.arange(n)
    labs = feat_refuge["labels"][ix].numpy()
    lgts = feat_refuge["logits"][ix].numpy()      # [n, num_classes]
    for i in range(n):
        row = {
            "case_id": case_ids[i],
            "dataset": "refuge",
            "split":   s,
            "is_ood":  0,
            "y_true":  int(labs[i]),
            "logit_0": float(lgts[i, 0]),          # normal
            "logit_1": float(lgts[i, 1]),           # glaucoma
        }
        for method in scores:
            row[method] = float(scores[method][("refuge", s)][i])
        rows.append(row)

# OOD datasets
for ood_name, feat in ood_sets.items():
    n = len(feat["logits"])
    case_ids = feat["case_id"] if "case_id" in feat else np.arange(n)
    labs = feat["labels"].numpy() if "labels" in feat else np.full(n, -1)
    lgts = feat["logits"].numpy()
    for i in range(n):
        row = {
            "case_id": case_ids[i],
            "dataset": ood_name,
            "split":   "all",
            "is_ood":  1,
            "y_true":  int(labs[i]),
            "logit_0": float(lgts[i, 0]),
            "logit_1": float(lgts[i, 1]),
        }
        for method in scores:
            row[method] = float(scores[method][(ood_name, "all")][i])
        rows.append(row)

df_ood = pd.DataFrame(rows)

In [55]:
# ── Standardize Mahalanobis using REFUGE train ──
train_mask = (df_ood["dataset"] == "refuge") & (df_ood["split"] == "train")
mu    = df_ood.loc[train_mask, "Mahalanobis"].mean()
sigma = df_ood.loc[train_mask, "Mahalanobis"].std()
df_ood["maha_risk"] = (df_ood["Mahalanobis"] - mu) / sigma

# ── Prediction uncertainty ──
logits_t = torch.tensor(df_ood[["logit_0", "logit_1"]].values)
probs = torch.softmax(logits_t, dim=1).numpy()
df_ood["prob_0"] = probs[:, 0]
df_ood["prob_1"] = probs[:, 1]
df_ood["pred"]   = (probs[:, 1] >= 0.5).astype(int)
df_ood["confidence"]  = probs.max(axis=1)
df_ood["uncertainty"] = 1.0 - df_ood["confidence"]

In [ ]:
metrics = {}
ood_names = ["origa", "chaksu"]

for method in scores:
    for ood_name in ood_names:
        s_tr = scores[method][("refuge", "train")]
        s_va = scores[method][("refuge", "val")]
        s_te = scores[method][("refuge", "test")]
        s_ood = scores[method][(ood_name, "all")]

        m = ood_metrics(s_te, s_ood, tau=99.0)
        val_tau = np.percentile(s_va, 99.0)
        m["FPR99_val_tau"] = float(np.mean(s_ood <= val_tau))*
        m["tau@99ID_val"]  = float(val_tau)
        metrics[(method, ood_name)] = m

metrics_df = pd.DataFrame(metrics).T
metrics_df.index = pd.MultiIndex.from_tuples(metrics_df.index, names=["Method", "OOD_dataset"])
metrics_df = metrics_df.sort_values(["OOD_dataset", "AUROC"], ascending=[True, False])
metrics_df

,,AUROC,AUPR_OOD,FPR99,tau@99ID,FPR99_val_tau,tau@99ID_val
Method,OOD_dataset,,,,,,
ViM,chaksu,0.977701,0.993726,0.163569,14.003281,0.132342,13.228146
Mahalanobis,chaksu,0.932468,0.983192,0.093680,7968.863281,0.093680,7972.955078
kNN,chaksu,0.784390,0.920133,0.862454,0.053772,0.906320,0.055919
Energy+ReAct,chaksu,0.756121,0.912444,0.797026,2.054438,0.992565,2.592132
MSP,chaksu,0.725746,0.861280,0.968030,0.454451,0.979182,0.470102
Entropy,chaksu,0.725745,0.861280,0.968030,0.688990,0.979182,0.691358
MaxLogit,chaksu,0.723559,0.856950,0.985874,-0.067355,0.988848,-0.054747
Energy,chaksu,0.722480,0.855025,0.985874,-0.653935,0.978439,-0.671073
ViM,origa,0.983046,0.992145,0.043077,14.003281,0.040000,13.228146


In [59]:
# ── Standardize Mahalanobis using REFUGE train ──
train_mask = (df_ood["dataset"] == "refuge") & (df_ood["split"] == "train")
mu    = df_ood.loc[train_mask, "Mahalanobis"].mean()
sigma = df_ood.loc[train_mask, "Mahalanobis"].std()
df_ood["maha_risk"] = (df_ood["Mahalanobis"] - mu) / sigma

mu_vim    = df_ood.loc[train_mask, "ViM"].mean()
sigma_vim = df_ood.loc[train_mask, "ViM"].std()
df_ood["vim_risk"] = (df_ood["ViM"] - mu_vim) / sigma_vim

# ── Prediction uncertainty ──
logits_t = torch.tensor(df_ood[["logit_0", "logit_1"]].values)
probs = torch.softmax(logits_t, dim=1).numpy()
df_ood["prob_0"] = probs[:, 0]
df_ood["prob_1"] = probs[:, 1]
df_ood["confidence"]  = probs.max(axis=1)
df_ood["uncertainty"] = 1.0 - df_ood["confidence"]

In [60]:
df_ood.shape

(3195, 22)

In [61]:
df_ood.groupby(["dataset"])["case_id"].count()

dataset
chaksu    1345
origa      650
refuge    1200
Name: case_id, dtype: int64

In [62]:
df_ood.to_csv(r"D:\Chaksu\ood_logits_all.csv", index=False)

In [66]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

def eval_logits(feat_dict, name="", split=None):
    """Run classification metrics on extracted features dict."""
    logits = feat_dict["logits"]
    y_true = feat_dict["labels"].numpy()
    
    # optional split filter
    if split is not None:
        mask = feat_dict["split"] == split
        logits = logits[mask]
        y_true = y_true[mask]
    
    probs = torch.softmax(logits, dim=1).numpy()
    p_pos = probs[:, 1]
    y_pred = (p_pos >= 0.5).astype(int)
    
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, p_pos) if len(np.unique(y_true)) > 1 else float("nan")
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    label = f"{name} [{split}]" if split else name
    print(f"\n{'='*50}")
    print(f"  {label}  (n={len(y_true)})")
    print(f"{'='*50}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Type I  (FPR): {fp/(fp+tn):.4f}")
    print(f"Type II (FNR): {fn/(fn+tp):.4f}")

# REFUGE — per split
for s in ("train", "val", "test"):
    eval_logits(feat_refuge, "REFUGE", split=None)

# ORIGA — per split
for s in ("train", "test"):
    eval_logits(feat_origa, "ORIGA", split=None)

for s in ("train", "test"):
    eval_logits(feat_cha, "CHAKSU", split=None)


  REFUGE  (n=1200)
Accuracy : 0.8925
Precision: 0.4815
Recall   : 0.9750
F1 score : 0.6446
ROC-AUC  : 0.9898
Confusion Matrix:
[[954 126]
 [  3 117]]
Type I  (FPR): 0.1167
Type II (FNR): 0.0250

  REFUGE  (n=1200)
Accuracy : 0.8925
Precision: 0.4815
Recall   : 0.9750
F1 score : 0.6446
ROC-AUC  : 0.9898
Confusion Matrix:
[[954 126]
 [  3 117]]
Type I  (FPR): 0.1167
Type II (FNR): 0.0250

  REFUGE  (n=1200)
Accuracy : 0.8925
Precision: 0.4815
Recall   : 0.9750
F1 score : 0.6446
ROC-AUC  : 0.9898
Confusion Matrix:
[[954 126]
 [  3 117]]
Type I  (FPR): 0.1167
Type II (FNR): 0.0250

  ORIGA  (n=650)
Accuracy : 0.5415
Precision: 0.3529
Recall   : 0.9286
F1 score : 0.5115
ROC-AUC  : 0.7955
Confusion Matrix:
[[196 286]
 [ 12 156]]
Type I  (FPR): 0.5934
Type II (FNR): 0.0714

  ORIGA  (n=650)
Accuracy : 0.5415
Precision: 0.3529
Recall   : 0.9286
F1 score : 0.5115
ROC-AUC  : 0.7955
Confusion Matrix:
[[196 286]
 [ 12 156]]
Type I  (FPR): 0.5934
Type II (FNR): 0.0714

  CHAKSU  (n=1345)
Accuracy 